In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd

In [3]:
data_dir = str(PROJECT_ROOT / "data" / "processed" / "TEP") + "/"
references_dir = str(PROJECT_ROOT / "references" / "TEP") + "/"
models_dir = str(PROJECT_ROOT / "models" / "TEP") + "/"
max_lag_selection_dir = models_dir + "max_lag_selection/"
weights_dir = models_dir + "max_lag_selection/weights/"
y_faultfree_dir = models_dir + "max_lag_selection/y/"
y_pred_faultfree_dir = models_dir + "max_lag_selection/y_pred/"
x_faulty_dir = models_dir + "moving_average_window_selection/X/"
y_faulty_dir = models_dir + "moving_average_window_selection/y/"
y_pred_faulty_dir = models_dir + "moving_average_window_selection/y_pred/"

In [4]:
import os

for directory in [models_dir, max_lag_selection_dir, weights_dir, y_faultfree_dir, y_pred_faultfree_dir, x_faulty_dir, y_faulty_dir, y_pred_faulty_dir]:
    os.makedirs(directory, exist_ok=True)

In [5]:
training_file_name = "TEP_Training.parquet"
adjacency_matrix_file_name = "TEGroundTruth.txt"

In [6]:
df = pd.read_parquet(data_dir + training_file_name)

In [7]:
n_faulty_samples = df[df["faultNumber"] != 0].shape[0]
df_faulty = df.iloc[-n_faulty_samples:]

n_samples_per_faulty_ts = df[df["faultNumber"] != 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faulty_ts = df[df["faultNumber"] != 0].groupby(["sample"]).size().max()

df_faulty = df_faulty.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [8]:
adjacency_matrix = np.loadtxt(references_dir + adjacency_matrix_file_name)

In [9]:
feature_names = df_faulty.columns
n_features = len(feature_names)
child_parents_pairs = {}

for child_idx in range(n_features):
    parents = []
    for feature_idx in range(n_features):
        if adjacency_matrix[feature_idx, child_idx] == 1:
            parents.append(feature_names[feature_idx])
    child_parents_pairs[feature_names[child_idx]] = parents

In [10]:
process_children = [child for child in child_parents_pairs.keys() if len(child_parents_pairs[child]) != 0 and int(child[1:]) < 23]
n_process_children = len(process_children)
process_child_parents_pairs = {child: child_parents_pairs[child] for child in process_children}

In [11]:
faultfree = df[df["faultNumber"] == 0]

n_faultfree_ts = faultfree["simulationRun"].nunique()

n_samples_per_faultfree_ts = (faultfree.groupby("simulationRun").size().max())

In [12]:
max_lag = 1

In [13]:
# Compute the sample covariance matrix of the residuals for the training data

n_samples_train = n_faultfree_ts * (n_samples_per_faultfree_ts - max_lag)
res_train = np.empty((n_samples_train, n_process_children))

for child, parents in process_child_parents_pairs.items():

    y_train = np.load(y_faultfree_dir + f"y_child_{child}_lag_{max_lag}.npy")
    y_train_pred = np.load(y_pred_faultfree_dir + f"y_pred_child_{child}_lag_{max_lag}.npy")          

    res_train[:, process_children.index(child)] = y_train - y_train_pred

sample_cov = (res_train.T @ res_train) / n_samples_train
sample_precision = np.linalg.inv(sample_cov)

np.save(max_lag_selection_dir + "sample_precision_train.npy", sample_precision)

In [14]:
n_samples_val = n_faulty_ts * (n_samples_per_faulty_ts - max_lag)
res_val = np.empty((n_samples_val, n_process_children))

for child, parents in process_child_parents_pairs.items():

    df_X_val = df_faulty[parents + [child]]
    y_val = df_faulty[child]

    df_X_val_lagged = pd.concat([df_X_val.shift(i).add_suffix(f"_lag{i}") for i in range(1, max_lag + 1)], axis=1)

    mask = []
    for ts_start_idx in range(0, n_faulty_samples, n_samples_per_faulty_ts):
        mask.extend(range(ts_start_idx + max_lag, ts_start_idx + n_samples_per_faulty_ts))

    df_X_val_lagged = df_X_val_lagged.iloc[mask]
    y_val = y_val.iloc[mask]

    X_val = np.ones((df_X_val_lagged.shape[0], df_X_val_lagged.shape[1] + 1))
    X_val[:, 1:] = df_X_val_lagged.values
    y_val = y_val.to_numpy()

    beta = np.load(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy")

    y_val_pred = X_val @ beta

    res_val[:, process_children.index(child)] = y_val - y_val_pred   

    np.save(x_faulty_dir + f"X_child_{child}.npy", X_val)
    np.save(y_faulty_dir + f"y_child_{child}.npy", y_val)
    np.save(y_pred_faulty_dir + f"y_pred_child_{child}_lag_{max_lag}.npy", y_val_pred)

In [15]:
scores_raw = np.mean(res_val @ sample_precision * res_val, axis=1)
scores_2d = scores_raw.reshape((n_faulty_ts, n_samples_per_faulty_ts - max_lag))
df_scores_2d = pd.DataFrame(scores_2d)

In [16]:
n_faultfree_samples_per_ts = 20
faultfree_idxs = range(n_faultfree_samples_per_ts - max_lag)

y_anomaly_2d = np.ones_like(scores_2d, dtype=int)
y_anomaly_2d[:, faultfree_idxs] = 0

In [17]:
min_window = 1
max_window = 8
min_consecutive_alarms = 5
max_consecutive_alarms = 8

windows = range(min_window, max_window + 1)
consecutive_alarms_list = range(min_consecutive_alarms, max_consecutive_alarms + 1)

y_anomaly_2d_red = y_anomaly_2d[:, max_window-1:]
idx_first_anomaly = n_faultfree_samples_per_ts - max_lag - max_window + 1

y_ts_anomaly_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_2d[:, 0] = y_anomaly_2d_red[:, :idx_first_anomaly].max(axis=1)
y_ts_anomaly_2d[:, 1] = y_anomaly_2d_red[:, idx_first_anomaly:].max(axis=1)
y_ts_anomaly = y_ts_anomaly_2d.reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample Accuracy",
    "Sample TPR",
    "Sample FPR",
    "Sample F1",
    "Trajectory-Phase Accuracy",
    "Trajectory-Phase TPR",
    "Trajectory-Phase FPR",
    "Trajectory-Phase F1",
    "Median Anomaly Detection Delay"
])

for w in windows:

    scores_2d_smoothed = df_scores_2d.T.rolling(window=w).mean().values.T
    scores_2d_smoothed = scores_2d_smoothed[:, max_window-1:]

    max_scores_smoothed = scores_2d_smoothed[:, :idx_first_anomaly].max(axis=1)
    threshold = np.max(max_scores_smoothed)

    raw_breaches = (scores_2d_smoothed > threshold).astype(int)

    for c in consecutive_alarms_list:
        if c > 1:
            y_anomaly_pred_2d = (
                pd.DataFrame(raw_breaches.T).rolling(window=c, min_periods=c).sum() == c
            ).astype(int).values.T
            
            y_anomaly_pred_2d = np.nan_to_num(y_anomaly_pred_2d, nan=0.0)

            eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
            eval_mask_2d[:, :c-1] = False
        else:
            y_anomaly_pred_2d = raw_breaches
            eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)

        y_anomaly = y_anomaly_2d_red.reshape(-1)
        y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
        eval_mask = eval_mask_2d.reshape(-1)

        y_anomaly_eval = y_anomaly[eval_mask]
        y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

        tp = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 1))
        fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
        fn = np.sum((y_anomaly_eval == 1) & (y_anomaly_pred_eval == 0))
        tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))

        accuracy = (tp + tn) / (tp + fp + fn + tn)
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

        y_ts_anomaly_pred_2d = np.empty((n_faulty_ts, 2))
        y_ts_anomaly_pred_2d[:, 0] = y_anomaly_pred_2d[:, :idx_first_anomaly].max(axis=1)

        faulty_phase_preds = y_anomaly_pred_2d[:, idx_first_anomaly:]
        y_ts_anomaly_pred_2d[:, 1] = faulty_phase_preds.max(axis=1)

        detected_mask = faulty_phase_preds.any(axis=1)
        if np.any(detected_mask):
            idx_first_anomaly_pred = faulty_phase_preds[detected_mask].argmax(axis=1)
            median_idx_first_anomaly_pred = np.median(idx_first_anomaly_pred)
        else:
            median_idx_first_anomaly_pred = np.nan
        
        y_ts_anomaly_pred = y_ts_anomaly_pred_2d.reshape(-1)

        tp_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 1))
        fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
        fn_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 0))
        tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

        accuracy_ts = (tp_ts + tn_ts) / (tp_ts + fp_ts + fn_ts + tn_ts)
        tpr_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
        fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0
        precision_ts = tp_ts / (tp_ts + fp_ts) if (tp_ts + fp_ts) > 0 else 0
        recall_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
        f1_ts = 2 * precision_ts * recall_ts / (precision_ts + recall_ts) if precision_ts + recall_ts > 0 else 0

        df_result = pd.DataFrame({
            "Window Size": [w],
            "Consecutive Alarms": [c],
            "Threshold": [threshold],
            "Sample Accuracy": [accuracy],
            "Sample TPR": [tpr],
            "Sample FPR": [fpr],
            "Sample F1": [f1],
            "Trajectory-Phase Accuracy": [accuracy_ts],
            "Trajectory-Phase TPR": [tpr_ts],
            "Trajectory-Phase FPR": [fpr_ts],
            "Trajectory-Phase F1": [f1_ts],
            "Median Anomaly Detection Delay": [median_idx_first_anomaly_pred],
        })

        df_results = pd.concat([df_results, df_result], ignore_index=True)

f1_col = "Trajectory-Phase F1"
delay_col = "Median Anomaly Detection Delay"

best_f1 = df_results[f1_col].max()
best_candidates = df_results.loc[df_results[f1_col].eq(best_f1)]
best_candidates = best_candidates.dropna(subset=[delay_col])
if best_candidates.empty:
    raise ValueError("No best-F1 configuration has a valid detection delay.")
best_idx = best_candidates[delay_col].idxmin()

best_window = df_results.loc[best_idx, "Window Size"]
best_cons_alarms = df_results.loc[best_idx, "Consecutive Alarms"]
best_threshold = df_results.loc[best_idx, "Threshold"]

print("Highest Trajectory-Phase F1:")
print(f"\tWindow Size = {best_window}")
print(f"\tConsecutive Alarms = {best_cons_alarms}")
print(f"\tThreshold = {best_threshold}")

with pd.option_context('display.max_rows', None):
    display(df_results.sort_values("Trajectory-Phase F1", ascending=False))

<ipykernel>\2936506547.py:117: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


Highest Trajectory-Phase F1:
	Window Size = 6
	Consecutive Alarms = 5
	Threshold = 1.7085420289572733


,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Median Anomaly Detection Delay
20,6,5,1.708542,0.521505,0.513530,0.0,0.678586,0.93065,0.8613,0.0,0.925482,16.0
28,8,5,1.686269,0.538395,0.530702,0.0,0.693410,0.92950,0.8590,0.0,0.924153,17.0
24,7,5,1.741776,0.523052,0.515103,0.0,0.679957,0.92830,0.8566,0.0,0.922762,17.0
29,8,6,1.686269,0.528433,0.521556,0.0,0.685556,0.92790,0.8558,0.0,0.922298,18.0
21,6,6,1.708542,0.509991,0.502845,0.0,0.669191,0.92760,0.8552,0.0,0.921949,18.0
25,7,6,1.741776,0.512369,0.505258,0.0,0.671324,0.92675,0.8535,0.0,0.920960,18.0
30,8,7,1.686269,0.518709,0.512693,0.0,0.677854,0.92610,0.8522,0.0,0.920203,19.0
16,5,5,1.876248,0.488860,0.480341,0.0,0.648960,0.92540,0.8508,0.0,0.919386,17.0
26,7,7,1.741776,0.501997,0.495771,0.0,0.662897,0.92330,0.8466,0.0,0.916928,20.0
31,8,8,1.686269,0.509214,0.504102,0.0,0.670303,0.92270,0.8454,0.0,0.916224,21.0
